# Demo 1: Parent-Paper CNN on GTZAN, then Evaluate on AI-Generated Music

This notebook ties together three pieces of your DS340 project:
- the parent paper PDF at `/Users/mingoosim/Downloads/Music Genre Classification using Deep Learning (1)-2.pdf`,
- the local GTZAN copy at `data/genres_original`,
- the local AI-generated music dataset at `data/suno-audio`.

Goal:
- use the same MFCC-based preprocessing and paper-style CNN from the replication code,
- train on the original GTZAN-style dataset,
- evaluate the trained model on AI-generated Suno tracks whose tags can be mapped into the GTZAN label space.

Important caveat for the report:
- the Suno dataset does **not** come with the same 10 clean labels as GTZAN,
- instead, this notebook uses a transparent keyword-mapping heuristic from Suno tags to GTZAN genres,
- only tracks with a single unambiguous mapped genre are kept for AI evaluation,
- for longer Suno tracks, only the first 30 seconds are used so preprocessing stays aligned with the paper/replication setup.

In [1]:
%pip install -q numpy pandas scikit-learn librosa soundfile matplotlib seaborn tensorflow datasets


[notice] A new release of pip is available: 25.0.1 -> 26.0.1
[notice] To update, run: /Library/Developer/CommandLineTools/usr/bin/python3 -m pip install --upgrade pip
Note: you may need to restart the kernel to use updated packages.


In [3]:
import json
import math
import os
import tempfile
from collections import Counter
from pathlib import Path

PROJECT_ROOT = Path.cwd()
MPLCONFIG_DIR = PROJECT_ROOT / ".mplconfig"
MPLCONFIG_DIR.mkdir(parents=True, exist_ok=True)
os.environ.setdefault("MPLCONFIGDIR", str(MPLCONFIG_DIR))

import librosa
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd

In [2]:
import seaborn as sns

In [ ]:
import tensorflow.keras as keras

In [ ]:
import sys
print(sys.executable)
print(sys.version)

import time
t = time.time()
import tensorflow.keras as keras
print("tensorflow.keras import time:", time.time() - t)

In [ ]:

from datasets import Audio, load_from_disk
from sklearn.metrics import accuracy_score, classification_report, confusion_matrix, f1_score
from sklearn.model_selection import train_test_split

In [ ]:
import json
import math
import os
import tempfile
from collections import Counter
from pathlib import Path

PROJECT_ROOT = Path.cwd()
MPLCONFIG_DIR = PROJECT_ROOT / ".mplconfig"
MPLCONFIG_DIR.mkdir(parents=True, exist_ok=True)
os.environ.setdefault("MPLCONFIGDIR", str(MPLCONFIG_DIR))

import librosa
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import seaborn as sns
import tensorflow.keras as keras
from datasets import Audio, load_from_disk
from sklearn.metrics import accuracy_score, classification_report, confusion_matrix, f1_score
from sklearn.model_selection import train_test_split

PAPER_PDF = Path("/Users/mingoosim/Downloads/Music Genre Classification using Deep Learning (1)-2.pdf")
GTZAN_RAW_DIR = PROJECT_ROOT / "data" / "genres_original"
AI_DATA_ROOT = PROJECT_ROOT / "data" / "suno-audio"

ARTIFACT_DIR = PROJECT_ROOT / "artifacts" / "demo_1"
ARTIFACT_DIR.mkdir(parents=True, exist_ok=True)

GTZAN_DATA_JSON = ARTIFACT_DIR / "gtzan_data.json"
MODEL_PATH = ARTIFACT_DIR / "paper_cnn.keras"
HISTORY_CSV = ARTIFACT_DIR / "gtzan_training_history.csv"
AI_JSON_PATH = ARTIFACT_DIR / "suno_ai_eval_mfcc.json"
AI_METADATA_CSV = ARTIFACT_DIR / "suno_ai_eval_metadata.csv"
AI_SEGMENT_RESULTS_CSV = ARTIFACT_DIR / "ai_segment_results.csv"
AI_TRACK_RESULTS_CSV = ARTIFACT_DIR / "ai_track_results.csv"

SAMPLE_RATE = 22050
TRACK_DURATION = 30
SAMPLES_PER_TRACK = SAMPLE_RATE * TRACK_DURATION
NUM_MFCC = 13
N_FFT = 2048
HOP_LENGTH = 512
NUM_SEGMENTS = 5

TEST_SIZE = 0.3
VALIDATION_SIZE = 0.2
LEARNING_RATE = 1e-4
EPOCHS = 30
BATCH_SIZE = 32
SEED = 42

REBUILD_GTZAN_JSON = not GTZAN_DATA_JSON.exists()
RETRAIN_MODEL = False
STRICT_SINGLE_LABEL = True
MAX_AI_TRACKS_PER_GENRE = 30
REBUILD_AI_FEATURES = False

keras.utils.set_random_seed(SEED)

print(f"Paper PDF exists: {PAPER_PDF.exists()} -> {PAPER_PDF}")
print(f"GTZAN raw folder exists: {GTZAN_RAW_DIR.exists()} -> {GTZAN_RAW_DIR}")
print(f"GTZAN MFCC JSON exists: {GTZAN_DATA_JSON.exists()} -> {GTZAN_DATA_JSON}")
print(f"AI data root exists: {AI_DATA_ROOT.exists()} -> {AI_DATA_ROOT}")
print(f"Artifacts will be written to: {ARTIFACT_DIR}")
print(f"Will rebuild GTZAN MFCC JSON: {REBUILD_GTZAN_JSON}")

if not GTZAN_DATA_JSON.exists() and not GTZAN_RAW_DIR.exists():
    raise FileNotFoundError("Need either the raw GTZAN folder or an existing GTZAN MFCC JSON file.")

if not AI_DATA_ROOT.exists():
    raise FileNotFoundError("Could not find the local Suno AI dataset folder.")

## Paper / replication settings used here

These settings are aligned with the replication repo you referenced:
- sample rate: `22050`
- track duration used for features: `30` seconds
- MFCC count: `13`
- FFT window: `2048`
- hop length: `512`
- segments per track: `5`
- optimizer: `Adam(learning_rate=1e-4)`
- loss: `sparse_categorical_crossentropy`
- metric: `accuracy`
- evaluation style for the original task: train/test split with `test_size=0.3`

For AI-generated music, the primary evaluation below is still **segment-level accuracy** so it stays comparable to the original segmented MFCC pipeline. A supplementary **track-level majority-vote** result is also reported because it is easier to explain in the project report.

In [ ]:
def extract_mfcc_segments_from_file(file_path, num_mfcc=NUM_MFCC, n_fft=N_FFT, hop_length=HOP_LENGTH, num_segments=NUM_SEGMENTS):
    signal, sr = librosa.load(file_path, sr=SAMPLE_RATE)
    samples_per_segment = int(SAMPLES_PER_TRACK / num_segments)
    expected_vectors = math.ceil(samples_per_segment / hop_length)

    segments = []
    for segment_idx in range(num_segments):
        start_sample = samples_per_segment * segment_idx
        finish_sample = start_sample + samples_per_segment

        mfcc = librosa.feature.mfcc(
            y=signal[start_sample:finish_sample],
            sr=sr,
            n_mfcc=num_mfcc,
            n_fft=n_fft,
            hop_length=hop_length,
        ).T

        if len(mfcc) == expected_vectors:
            segments.append(mfcc)

    return segments


def save_mfcc_from_folder(dataset_path, json_path, num_mfcc=NUM_MFCC, n_fft=N_FFT, hop_length=HOP_LENGTH, num_segments=NUM_SEGMENTS):
    dataset_path = Path(dataset_path)
    data = {"mapping": [], "labels": [], "mfcc": []}

    for genre_idx, genre_dir in enumerate(sorted(p for p in dataset_path.iterdir() if p.is_dir())):
        data["mapping"].append(genre_dir.name)
        print(f"Processing GTZAN genre: {genre_dir.name}")

        for audio_file in sorted(genre_dir.iterdir()):
            if audio_file.suffix.lower() not in {".wav", ".mp3", ".au", ".flac", ".ogg", ".m4a"}:
                continue

            try:
                segments = extract_mfcc_segments_from_file(audio_file)
            except Exception as exc:
                print(f"SKIP (cannot read): {audio_file} | {type(exc).__name__}: {exc}")
                continue

            for mfcc in segments:
                data["mfcc"].append(mfcc.tolist())
                data["labels"].append(genre_idx)

    with open(json_path, "w") as fp:
        json.dump(data, fp, indent=2)

    return data


def load_mfcc_json(json_path):
    with open(json_path, "r") as fp:
        data = json.load(fp)

    X = np.array(data["mfcc"], dtype=np.float32)
    y = np.array(data["labels"], dtype=np.int64)
    return data, X, y


def prepare_datasets_from_json(json_path, test_size=TEST_SIZE, validation_size=VALIDATION_SIZE, random_state=SEED):
    data, X, y = load_mfcc_json(json_path)

    X_train, X_test, y_train, y_test = train_test_split(
        X, y, test_size=test_size, random_state=random_state
    )
    X_train, X_val, y_train, y_val = train_test_split(
        X_train, y_train, test_size=validation_size, random_state=random_state
    )

    X_train = X_train[..., np.newaxis]
    X_val = X_val[..., np.newaxis]
    X_test = X_test[..., np.newaxis]

    return data["mapping"], X_train, X_val, X_test, y_train, y_val, y_test


def build_paper_cnn(input_shape, num_classes):
    model = keras.Sequential()

    model.add(keras.layers.Conv2D(32, (3, 3), activation="relu", input_shape=input_shape))
    model.add(keras.layers.MaxPooling2D((3, 3), strides=(2, 2), padding="same"))
    model.add(keras.layers.BatchNormalization())

    model.add(keras.layers.Conv2D(32, (3, 3), activation="relu"))
    model.add(keras.layers.MaxPooling2D((3, 3), strides=(2, 2), padding="same"))
    model.add(keras.layers.BatchNormalization())

    model.add(keras.layers.Conv2D(32, (2, 2), activation="relu"))
    model.add(keras.layers.MaxPooling2D((2, 2), strides=(2, 2), padding="same"))
    model.add(keras.layers.BatchNormalization())

    model.add(keras.layers.Flatten())
    model.add(keras.layers.Dense(64, activation="relu"))
    model.add(keras.layers.Dropout(0.3))
    model.add(keras.layers.Dense(num_classes, activation="softmax"))

    return model


def plot_confusion_matrix(y_true, y_pred, label_names, title):
    cm = confusion_matrix(y_true, y_pred, labels=list(range(len(label_names))))
    plt.figure(figsize=(8, 6))
    sns.heatmap(cm, annot=True, fmt="d", cmap="Blues", xticklabels=label_names, yticklabels=label_names)
    plt.xlabel("Predicted")
    plt.ylabel("True")
    plt.title(title)
    plt.tight_layout()
    plt.show()

In [ ]:
if REBUILD_GTZAN_JSON:
    print("Rebuilding GTZAN MFCC JSON from raw audio...")
    save_mfcc_from_folder(GTZAN_RAW_DIR, GTZAN_DATA_JSON)

label_names, X_train, X_val, X_test, y_train, y_val, y_test = prepare_datasets_from_json(GTZAN_DATA_JSON)
genre_to_id = {genre: idx for idx, genre in enumerate(label_names)}

print("Label order used by the model:", label_names)
print("GTZAN split shapes:")
print("  X_train:", X_train.shape, "y_train:", y_train.shape)
print("  X_val:  ", X_val.shape, "y_val:  ", y_val.shape)
print("  X_test: ", X_test.shape, "y_test: ", y_test.shape)

if MODEL_PATH.exists() and not RETRAIN_MODEL:
    print(f"Loading existing model from {MODEL_PATH}")
    model = keras.models.load_model(MODEL_PATH)
    history_df = pd.read_csv(HISTORY_CSV) if HISTORY_CSV.exists() else pd.DataFrame()
else:
    model = build_paper_cnn(X_train.shape[1:], len(label_names))
    optimiser = keras.optimizers.Adam(learning_rate=LEARNING_RATE)
    model.compile(optimizer=optimiser, loss="sparse_categorical_crossentropy", metrics=["accuracy"])

    history = model.fit(
        X_train,
        y_train,
        validation_data=(X_val, y_val),
        batch_size=BATCH_SIZE,
        epochs=EPOCHS,
        verbose=1,
    )

    model.save(MODEL_PATH)
    history_df = pd.DataFrame(history.history)
    history_df.to_csv(HISTORY_CSV, index=False)

display(history_df.tail())

gtzan_test_loss, gtzan_test_acc = model.evaluate(X_test, y_test, verbose=0)
gtzan_test_pred = model.predict(X_test, verbose=0).argmax(axis=1)

print(f"GTZAN test accuracy: {gtzan_test_acc:.4f}")
print(classification_report(y_test, gtzan_test_pred, labels=list(range(len(label_names))), target_names=label_names, zero_division=0))
plot_confusion_matrix(y_test, gtzan_test_pred, label_names, "GTZAN Test Confusion Matrix")

## Build an AI-generated evaluation subset

The Suno dataset uses free-form tags instead of GTZAN labels, so we need a mapping layer.

Strategy used here:
- normalize Suno tags to lower-case comma-separated tokens,
- map only a small set of explicit tag aliases into the GTZAN label set,
- keep only tracks with exactly one mapped GTZAN genre when `STRICT_SINGLE_LABEL = True`,
- optionally cap the number of tracks per genre with `MAX_AI_TRACKS_PER_GENRE` to keep the demo runnable.

This is the part you should describe carefully in the project report because it is the main methodological difference between the original GTZAN setup and the new AI-generated evaluation.

In [ ]:
GENRE_ALIASES = {
    "blues": {"blues"},
    "classical": {"classical", "orchestra", "orchestral", "symphony"},
    "country": {"country"},
    "disco": {"disco"},
    "hiphop": {"hip hop", "hiphop", "rap"},
    "jazz": {"jazz"},
    "metal": {"metal"},
    "pop": {"pop", "synthpop"},
    "reggae": {"reggae", "roots reggae", "reggae fusion", "ragga"},
    "rock": {"rock", "hard rock", "alternative rock", "symphonic rock"},
}


def normalize_tags(raw_tags):
    return {
        " ".join(part.strip().lower().split())
        for part in str(raw_tags).split(",")
        if part and part.strip()
    }


def map_tags_to_gtzan_genre(raw_tags, strict_single_label=True):
    tags = normalize_tags(raw_tags)
    matched = []

    for genre, aliases in GENRE_ALIASES.items():
        if any(alias in tags for alias in aliases):
            matched.append(genre)

    matched = sorted(set(matched))
    if len(matched) == 1:
        return matched[0]
    if strict_single_label:
        return None
    return matched[0] if matched else None


def discover_loadable_suno_batches(ai_root):
    loadable = []
    failed = []

    for batch_dir in sorted(p for p in ai_root.iterdir() if p.is_dir() and p.name.startswith("batch_")):
        try:
            ds = load_from_disk(str(batch_dir))
            loadable.append({"batch": batch_dir.name, "rows": ds.num_rows})
        except Exception as exc:
            failed.append({"batch": batch_dir.name, "error": str(exc)})

    return pd.DataFrame(loadable), pd.DataFrame(failed)


def collect_ai_eval_metadata(ai_root, max_ai_tracks_per_genre=MAX_AI_TRACKS_PER_GENRE, strict_single_label=STRICT_SINGLE_LABEL):
    loadable_df, failed_df = discover_loadable_suno_batches(ai_root)
    records = []

    for batch_name in loadable_df["batch"]:
        batch_dir = ai_root / batch_name
        ds = load_from_disk(str(batch_dir))
        meta = ds.select_columns(["id", "title", "tags", "prompt", "duration", "model_name", "status"])

        for row_index in range(len(meta)):
            row = meta[row_index]
            mapped_genre = map_tags_to_gtzan_genre(row["tags"], strict_single_label=strict_single_label)
            if mapped_genre is None:
                continue

            records.append({
                "batch": batch_name,
                "row_index": row_index,
                "track_id": str(row["id"]),
                "title": row["title"],
                "tags": row["tags"],
                "prompt": row["prompt"],
                "duration": row["duration"],
                "model_name": row["model_name"],
                "status": row["status"],
                "mapped_genre": mapped_genre,
            })

    metadata_df = pd.DataFrame(records)
    metadata_df = metadata_df.sort_values(["mapped_genre", "batch", "row_index"]).reset_index(drop=True)

    if max_ai_tracks_per_genre is not None and not metadata_df.empty:
        metadata_df = (
            metadata_df.groupby("mapped_genre", group_keys=False)
            .head(max_ai_tracks_per_genre)
            .reset_index(drop=True)
        )

    return loadable_df, failed_df, metadata_df


def extract_mfcc_segments_from_audio_bytes(audio_bytes, suffix):
    with tempfile.NamedTemporaryFile(suffix=suffix or ".mp3", delete=True) as tmp_file:
        tmp_file.write(audio_bytes)
        tmp_file.flush()
        return extract_mfcc_segments_from_file(tmp_file.name)


def build_ai_eval_json(metadata_df, ai_root, output_json_path):
    data = {
        "mapping": label_names,
        "labels": [],
        "mfcc": [],
        "track_ids": [],
        "track_titles": [],
        "track_genres": [],
        "source_batches": [],
        "row_indices": [],
    }

    skipped_tracks = []

    for batch_name, batch_rows in metadata_df.groupby("batch"):
        batch_ds = load_from_disk(str(ai_root / batch_name)).cast_column("audio", Audio(decode=False))

        for record in batch_rows.to_dict("records"):
            row = batch_ds[int(record["row_index"])]
            audio_info = row["audio"]
            suffix = Path(audio_info.get("path", "track.mp3")).suffix or ".mp3"

            try:
                segments = extract_mfcc_segments_from_audio_bytes(audio_info["bytes"], suffix)
            except Exception as exc:
                skipped_tracks.append({
                    "track_id": record["track_id"],
                    "title": record["title"],
                    "reason": f"{type(exc).__name__}: {exc}",
                })
                continue

            if not segments:
                skipped_tracks.append({
                    "track_id": record["track_id"],
                    "title": record["title"],
                    "reason": "No valid MFCC segments extracted",
                })
                continue

            label_id = genre_to_id[record["mapped_genre"]]
            for mfcc in segments:
                data["mfcc"].append(mfcc.tolist())
                data["labels"].append(int(label_id))
                data["track_ids"].append(str(record["track_id"]))
                data["track_titles"].append(str(record["title"]))
                data["track_genres"].append(str(record["mapped_genre"]))
                data["source_batches"].append(str(record["batch"]))
                data["row_indices"].append(int(record["row_index"]))

    with open(output_json_path, "w") as fp:
        json.dump(data, fp)

    return data, pd.DataFrame(skipped_tracks)


def load_ai_eval_json(json_path):
    with open(json_path, "r") as fp:
        data = json.load(fp)

    X = np.array(data["mfcc"], dtype=np.float32)[..., np.newaxis]
    y = np.array(data["labels"], dtype=np.int64)
    return data, X, y

In [ ]:
loadable_batches_df, failed_batches_df, ai_metadata_df = collect_ai_eval_metadata(AI_DATA_ROOT)

print(f"Loadable Suno batches: {len(loadable_batches_df)}")
print(f"Failed Suno batches:   {len(failed_batches_df)}")
print(f"Mapped AI tracks kept: {len(ai_metadata_df)}")

display(loadable_batches_df.head())
if not failed_batches_df.empty:
    display(failed_batches_df.head(10))

genre_counts = ai_metadata_df["mapped_genre"].value_counts().sort_index()
display(genre_counts.to_frame("tracks"))

ai_metadata_df.to_csv(AI_METADATA_CSV, index=False)
print(f"Saved AI metadata to: {AI_METADATA_CSV}")

if AI_JSON_PATH.exists() and not REBUILD_AI_FEATURES:
    print(f"Loading cached AI feature JSON from {AI_JSON_PATH}")
    ai_eval_data, skipped_tracks_df = load_ai_eval_json(AI_JSON_PATH)[0], pd.DataFrame()
else:
    print("Building AI evaluation MFCC JSON...")
    ai_eval_data, skipped_tracks_df = build_ai_eval_json(ai_metadata_df, AI_DATA_ROOT, AI_JSON_PATH)

print(f"AI feature JSON path: {AI_JSON_PATH}")
print(f"AI segment samples:   {len(ai_eval_data['labels'])}")
if not skipped_tracks_df.empty:
    display(skipped_tracks_df.head())

In [ ]:
ai_eval_data, X_ai, y_ai = load_ai_eval_json(AI_JSON_PATH)
ai_pred = model.predict(X_ai, verbose=0).argmax(axis=1)

ai_segment_acc = accuracy_score(y_ai, ai_pred)
ai_segment_macro_f1 = f1_score(y_ai, ai_pred, average="macro")

print(f"AI segment-level accuracy:  {ai_segment_acc:.4f}")
print(f"AI segment-level macro F1: {ai_segment_macro_f1:.4f}")
print(classification_report(y_ai, ai_pred, labels=list(range(len(label_names))), target_names=label_names, zero_division=0))
plot_confusion_matrix(y_ai, ai_pred, label_names, "AI Music Segment-Level Confusion Matrix")

segment_results_df = pd.DataFrame({
    "track_id": ai_eval_data["track_ids"],
    "title": ai_eval_data["track_titles"],
    "true_label_id": y_ai,
    "pred_label_id": ai_pred,
    "true_label": [label_names[idx] for idx in y_ai],
    "pred_label": [label_names[idx] for idx in ai_pred],
    "source_batch": ai_eval_data["source_batches"],
    "row_index": ai_eval_data["row_indices"],
})
segment_results_df.to_csv(AI_SEGMENT_RESULTS_CSV, index=False)

track_results_df = (
    segment_results_df.groupby(["track_id", "title", "true_label"], as_index=False)
    .agg(
        true_label_id=("true_label_id", "first"),
        pred_label_id=("pred_label_id", lambda series: Counter(series).most_common(1)[0][0]),
        num_segments=("pred_label_id", "size"),
    )
)
track_results_df["pred_label"] = track_results_df["pred_label_id"].map(lambda idx: label_names[idx])
track_results_df.to_csv(AI_TRACK_RESULTS_CSV, index=False)

ai_track_acc = accuracy_score(track_results_df["true_label_id"], track_results_df["pred_label_id"])
ai_track_macro_f1 = f1_score(track_results_df["true_label_id"], track_results_df["pred_label_id"], average="macro")

print(f"AI track-level accuracy:   {ai_track_acc:.4f}")
print(f"AI track-level macro F1:  {ai_track_macro_f1:.4f}")
display(track_results_df.head(10))
plot_confusion_matrix(track_results_df["true_label_id"], track_results_df["pred_label_id"], label_names, "AI Music Track-Level Confusion Matrix")

print(f"Saved segment-level results to: {AI_SEGMENT_RESULTS_CSV}")
print(f"Saved track-level results to:   {AI_TRACK_RESULTS_CSV}")

## What to discuss in the report

Suggested framing for the write-up:
- The training pipeline follows the paper-style replication: GTZAN, MFCCs, fixed FFT/hop settings, segmented inputs, and the same CNN family.
- The new experiment changes **only the evaluation domain**: real human-labeled genre tracks versus AI-generated tracks from Suno.
- The biggest methodological limitation is not the CNN, but the label bridge from Suno tags to GTZAN genres.
- Report both segment-level and track-level AI results, and explicitly state how many Suno tracks were usable after the mapping filter.
- If AI performance drops relative to GTZAN test accuracy, that supports the idea that genre classifiers trained on traditional datasets may not transfer cleanly to AI-generated music.